# Subtask 2 – Maximum Entropy Model

This notebook trains a Maximum Entropy model for Subtask 2 and generates the prediction file required for external evaluation.

## 1. Imports

In [ ]:
# Standard
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.utils.seed import set_seed
from src.subtask2a.models.maxent import MaxEnt
from src.subtask2a.models.autoencoder import BinaryAutoencoder

In [ ]:
set_seed(42)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Data Loading

In [ ]:
# Training data
df_train = pd.read_csv("data/subtask2a/train_data_enriched_1311.csv")
df_train = df_train.dropna(subset=["vector_10_binary_llm", "valence", "arousal"])

# Test template (authoritative source of rows)
df_test_template = pd.read_csv("data/subtask2a/subtask2a-template.csv")

# Identify which users belong to test
test_users = set(df_test_template["user_id"])

# Gold labels (evaluation only)
df_test_labels = pd.read_csv(
    "data/subtask2a/test_labels_subtask2a_and_2b.csv"
)

In [ ]:
train_parts = []
test_last_rows = []

for user_id, g in df_train.groupby("user_id"):

    g = g.sort_values("timestamp")

    if user_id in test_users:

        # keep all except last for training
        train_parts.append(g.iloc[:-1])

        # keep last row separately
        test_last_rows.append(g.iloc[-1:])

    else:
        # full history for training
        train_parts.append(g)


In [ ]:
train_df = pd.concat(train_parts).reset_index(drop=True)
test_df  = pd.concat(test_last_rows).reset_index(drop=True)


test_df = df_test_template.merge(
    test_df,
    on=["user_id"],
    how="left"
)


In [ ]:
print("Template rows:", len(df_test_template))
print("Constructed test_df rows:", len(test_df))
print("Unique users in constructed test_df:", test_df["user_id"].nunique())

## 3. Feature Builders

In [ ]:
LLM_LEVELS = [0, 1, 2, 3, 4, 5]
LLM_DIM_PER_POS = len(LLM_LEVELS) 
LLM_POSITIONS = 10
LLM_INPUT_DIM = LLM_POSITIONS * LLM_DIM_PER_POS

In [ ]:
def one_hot_llm_vector(vec_10):
    """
    vec_10: iterable of length 10, values in {0..5}
    returns: 60-dim binary vector
    """
    out = np.zeros(LLM_INPUT_DIM, dtype=np.float32)
    for i, v in enumerate(vec_10):
        idx = i * LLM_DIM_PER_POS + LLM_LEVELS.index(int(v))
        out[idx] = 1.0
    return out

def parse_llm_vector(x):
    if isinstance(x, (list, np.ndarray)):
        return list(map(int, x))
    if isinstance(x, str):
        return [int(v) for v in x.split(",")]
    raise ValueError(f"Unexpected type: {type(x)}")

In [ ]:
train_df["vector_10_binary_llm"] = (
    train_df["vector_10_binary_llm"].apply(parse_llm_vector)
)

test_df["vector_10_binary_llm"] = (
    test_df["vector_10_binary_llm"].apply(parse_llm_vector)
)

In [ ]:
llm_train_60 = np.stack(
    train_df["vector_10_binary_llm"].apply(one_hot_llm_vector).values
)

llm_test_60 = np.stack(
    test_df["vector_10_binary_llm"].apply(one_hot_llm_vector).values
)

## 4. Autoencoder

In [ ]:
X_train_ae = torch.tensor(llm_train_60, dtype=torch.float32, device=device)

In [ ]:
ae = BinaryAutoencoder(
    input_dim=60,
    hidden_dim=32,
    latent_dim=10
).to(device)

optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
criterion = nn.BCELoss()

In [ ]:
def train_autoencoder(
    model,
    X,
    epochs=50,
    batch_size=256,
    verbose=True
):
    model.train()
    N = X.size(0)

    for epoch in range(epochs):
        perm = torch.randperm(N)
        total_loss = 0.0

        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            batch = X[idx]

            optimizer.zero_grad()
            x_hat, _ = model(batch)
            loss = criterion(x_hat, batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch.size(0)

        avg_loss = total_loss / N
        if verbose:
            print(f"Epoch {epoch+1:03d} | Recon Loss: {avg_loss:.6f}")

    return model


ae = train_autoencoder(ae, X_train_ae, epochs=200)

## 5. Encode Latent States

In [ ]:
@torch.no_grad()
def compute_z(llm_array, autoencoder, device=device):
    X = torch.tensor(llm_array, dtype=torch.float32, device=device)
    z_bin, z_logits = autoencoder.encode(X)
    return z_bin.cpu().numpy(), z_logits.cpu().numpy()

In [ ]:
ae.eval()
ae.to(device)

Z_train_bin, Z_train_logits = compute_z(llm_train_60, ae)
Z_test_bin,  Z_test_logits  = compute_z(llm_test_60, ae)

In [ ]:
train_df["z"] = list(Z_train_bin)
test_df["z"]  = list(Z_test_bin)

train_df["r"] = train_df["z"].apply(lambda z: int(z.sum()))
test_df["r"]  = test_df["z"].apply(lambda z: int(z.sum()))

In [ ]:
def compute_s_from_r(r, tau=0):
    r = r.values
    s = np.zeros(len(r), dtype=float)

    for i in range(len(r) - 1):
        dr = r[i+1] - r[i]
        if dr > tau:
            s[i] = +1
        elif dr < -tau:
            s[i] = -1
        else:
            s[i] = 0

    s[-1] = np.nan
    return s


In [ ]:
train_df = train_df.sort_values(["user_id", "timestamp", "text_id"])
test_df = test_df.sort_values(["user_id", "timestamp", "text_id"])

train_df["s"] = (
    train_df.groupby("user_id")["r"]
      .transform(compute_s_from_r))

test_df["s"] = (
    test_df.groupby("user_id")["r"]
      .transform(compute_s_from_r))

## 6. Build MaxEnt Training Matrix

In [ ]:
VAL_MAP = {-2:0, -1:1, 0:2, 1:3, 2:4}
ARO_MAP = {0:0, 1:1, 2:2}
DV_MAP  = {-4:0, -3:1, -2:2, -1:3, 0:4, 1:5, 2:6, 3:7, 4:8}
DA_MAP  = {-2:0, -1:1, 0:2, 1:3, 2:4}
S_MAP = {-1: 0, 0: 1, 1: 2}
S_VALUES = [-1, 0, 1]


In [ ]:
import torch
import numpy as np
from itertools import product

def one_hot(index, size):
    v = np.zeros(size, dtype=np.float32)
    v[index] = 1.0
    return v

def encode_state(v, a, s, dv, da):
    return np.concatenate([
        one_hot(VAL_MAP[v], 5),
        one_hot(ARO_MAP[a], 3),
        one_hot(S_MAP[s],   3),
        one_hot(DV_MAP[dv], 9),
        one_hot(DA_MAP[da], 5),
    ])

def extract_observed_transitions(df):

    transitions = set()

    for user_id, g in df.groupby("user_id"):
        g = g.sort_values("timestamp")

        for i in range(len(g) - 1):

            v_t  = g.iloc[i]["valence"]
            a_t  = g.iloc[i]["arousal"]
            s_t  = g.iloc[i]["s"]

            v_t1 = g.iloc[i+1]["valence"]
            a_t1 = g.iloc[i+1]["arousal"]

            dv = v_t1 - v_t
            da = a_t1 - a_t

            transitions.add((v_t, a_t, s_t, dv, da))

    return transitions


def generate_valid_states_from_train(train_df):

    observed = extract_observed_transitions(train_df)

    states = []
    meta = []

    for (v, a, s, dv, da) in observed:
        states.append(encode_state(v, a, s, dv, da))
        meta.append((v, a, s, dv, da))

    states = np.array(states, dtype=np.float32)
    states = torch.from_numpy(states)

    return states, meta


In [ ]:
states, meta = generate_valid_states_from_train(train_df)

In [ ]:
X_train = []

for user_id, g in train_df.groupby("user_id"):
    g = g.sort_values("timestamp").reset_index(drop=True)

    for i in range(len(g) - 1):

        v_t  = g.loc[i,   "valence"]
        a_t  = g.loc[i,   "arousal"]
        v_t1 = g.loc[i+1, "valence"]
        a_t1 = g.loc[i+1, "arousal"]

        dv = v_t1 - v_t
        da = a_t1 - a_t
        s_t = g.loc[i, "s"]

        if np.isnan(s_t):
            continue

        if dv not in DV_MAP or da not in DA_MAP or s_t not in S_MAP:
            continue

        x = np.concatenate([
            one_hot(VAL_MAP[v_t], 5),
            one_hot(ARO_MAP[a_t], 3),
            one_hot(S_MAP[int(s_t)], 3),
            one_hot(DV_MAP[dv],   9),
            one_hot(DA_MAP[da],   5),
        ])

        X_train.append(x)

X_train = np.array(X_train, dtype=np.float32)

In [ ]:
dv_train = []
da_train = []

for x in X_train:
    # extract dv and da from one-hot encoding
    dv_index = np.argmax(x[5+3+3 : 5+3+3+9])
    da_index = np.argmax(x[5+3+3+9 : ])

    dv_train.append(list(DV_MAP.keys())[list(DV_MAP.values()).index(dv_index)])
    da_train.append(list(DA_MAP.keys())[list(DA_MAP.values()).index(da_index)])



## 9. Train MaxEnt

In [ ]:
model = MaxEnt(states, device='cuda')
model.fit(X_train, lr=1e-3, steps=10000, lambda_=0.5)

## 10. Inference

In [ ]:
def conditional_delta_distribution(model, meta, v_t, a_t):
    """
    Returns a dictionary:
    {(dv, da): probability}
    conditioned on (v_t, a_t)
    """

    model.eval()

    with torch.no_grad():
        probs, _ = model._compute_probabilities()
        probs = probs.cpu().numpy()

    dist = {}

    for idx, (v, a, s, dv, da) in enumerate(meta):
        if v == v_t and a == a_t:
            dist[(dv, da)] = probs[idx]

    # Normalize (important!)
    total = sum(dist.values())
    if total > 0:
        for k in dist:
            dist[k] /= total

    return dist

In [ ]:
def inference_subtask2a(model, meta, v_t, a_t):
    dist = conditional_delta_distribution(model, meta, v_t, a_t)

    if not dist:
        return 0.0, 0.0

    exp_dv = sum(dv * p for (dv, da), p in dist.items())
    exp_da = sum(da * p for (dv, da), p in dist.items())

    exp_dv = np.clip(exp_dv, -2, 2)
    exp_da = np.clip(exp_da, -2, 2)

    return exp_dv, exp_da

In [ ]:
test_eval = (
    test_df[["user_id", "valence", "arousal"]]
    .merge(
        df_test_labels[["user_id",
                        "state_change_valence",
                        "state_change_arousal"]],
        on="user_id"
    )
)

pred_dv = []
pred_da = []
true_dv = []
true_da = []

for _, row in test_eval.iterrows():

    v_t = row["valence"]
    a_t = row["arousal"]

    dv_true = row["state_change_valence"]
    da_true = row["state_change_arousal"]

    dv_hat, da_hat = inference_subtask2a(model, meta, v_t, a_t)

    pred_dv.append(dv_hat)
    pred_da.append(da_hat)
    true_dv.append(dv_true)
    true_da.append(da_true)


In [ ]:
submission = df_test_template.copy()

test_lookup = test_eval.set_index("user_id")

for i, row in submission.iterrows():

    user_id = row["user_id"]
    row_data = test_lookup.loc[user_id]

    v_t = row_data["valence"]
    a_t = row_data["arousal"]

    dv_hat, da_hat = inference_subtask2a(model, meta, v_t, a_t)

    dv_hat = int(round(np.clip(dv_hat, -4, 4)))
    da_hat = int(round(np.clip(da_hat, -2, 2)))

    submission.loc[i, "pred_state_change_valence"] = dv_hat
    submission.loc[i, "pred_state_change_arousal"] = da_hat

In [ ]:
submission.to_csv("results/subtask2a/subtask2a_predictions.csv", index=False)